# Taller 2 — Notebook 06: Interpretabilidad del modelo

**MINE-4101 · Predicción de churn para FinAlpes** · **Integrantes:** Daniel Triviño · Nicolás Bedoya

Entregable 4. Responde una pregunta que las métricas no responden: **¿por qué el modelo predice lo que predice?**
Sigue los pasos de la *Práctica 5: Interpretabilidad* del curso
([PracticasMachineLearning](https://datascience-uniandes.github.io/PracticasMachineLearning/talleres/practica-5/)),
que usa este mismo conjunto de datos. Se adapta al modelo elegido en el `04`, un XGBoost dentro de un pipeline, en
lugar del Random Forest de la práctica.

1. **Gráficos ICE** (comportamiento global y heterogeneidad entre clientes).
2. **LIME** (explicaciones locales de predicciones individuales, con su fidelidad y estabilidad).
3. **Valores SHAP** (explicaciones locales que, reunidas, dan una explicación global).
4. Referencia intrínsecamente interpretable (coeficientes de la regresión logística) y comparación de las técnicas.

> Ejecutar después de `04_modelado` (y preferiblemente del `05`).

In [ ]:
import json
import os

import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings('ignore', category=TqdmWarning)   # aviso de barra de progreso al importar shap
import shap
from lime.lime_tabular import LimeTabularExplainer
from matplotlib.lines import Line2D
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.inspection import PartialDependenceDisplay, partial_dependence
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid')

RUTA_PROCESADOS = '../data/processed'
RUTA_MODELOS = f'{RUTA_PROCESADOS}/modelos'
RUTA_FIGS = '../figs'
os.makedirs(RUTA_FIGS, exist_ok=True)

SEMILLA = 42

## 1. Modelo, datos y función de predicción

Las técnicas de interpretabilidad cambian los valores de las variables y observan cómo cambia la predicción. Por eso
deben trabajar con las **variables originales** (edad en años, saldo en euros), no con las columnas ya imputadas,
escaladas y codificadas que recibe el XGBoost por dentro. Como el modelo del `04` es un pipeline que recibe las
variables originales, la función de predicción solo tiene que convertir lo que le llega en un `DataFrame` y devolver la
probabilidad de abandono (Práctica 5, Actividad 1, paso 1).

In [ ]:
df = pd.read_parquet(f'{RUTA_PROCESADOS}/churn_particion.parquet')
with open(f'{RUTA_MODELOS}/seleccion.json', encoding='utf-8') as f:
    seleccion = json.load(f)

OBJETIVO = 'churn'
VARIABLES_MODELO = seleccion['variables_modelo']
CATEGORICAS = ['products_number', 'credit_card', 'active_member']   # discretas o binarias
modelo = joblib.load(f'{RUTA_MODELOS}/xgboost.joblib')

X = df[VARIABLES_MODELO].astype(float)
y = df[OBJETIVO]
es_prueba = (df['particion'] == 'prueba').to_numpy()
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]

def predecir_abandono(X):
    """Recibe las variables originales (DataFrame o arreglo de NumPy) y devuelve la probabilidad de abandono."""
    X = pd.DataFrame(np.asarray(X, dtype=float), columns=VARIABLES_MODELO)
    return modelo.predict_proba(X)[:, 1]

# Comprobación: la función predice exactamente lo mismo que el modelo
assert np.allclose(predecir_abandono(X_test.to_numpy()), modelo.predict_proba(X_test)[:, 1])
print(f'Modelo: {seleccion["modelo_elegido"]} {seleccion["hiperparametros"][seleccion["modelo_elegido"]]}')
print(f'Pasos del pipeline: {[nombre for nombre, _ in modelo.steps]}  |  la función reproduce sus predicciones en prueba')

**Muestras de trabajo.** LIME y los gráficos ICE no admiten valores faltantes en los registros que perturban. Por eso
las muestras se toman de clientes **sin faltantes**, que son \~89 % de cada conjunto. El modelo sí admite faltantes,
porque los imputa por dentro.

- **ICE:** 200 clientes de prueba.
- **SHAP:** 300 clientes de prueba, con un fondo de 100 clientes de entrenamiento.
- **Casos individuales:** salen de la muestra de SHAP, para poder explicar los mismos clientes con LIME y con SHAP.

El umbral de decisión es el mismo del `05`: el que maximiza el F1 en validación cruzada sobre entrenamiento. Se
recalcula aquí con las mismas particiones.

In [ ]:
X_test_completo = X_test.dropna()
X_train_completo = X_train.dropna()
print(f'Clientes sin faltantes: prueba {len(X_test_completo):,} de {len(X_test):,} | '
      f'entrenamiento {len(X_train_completo):,} de {len(X_train):,}')

X_muestra = X_test_completo.sample(300, random_state=SEMILLA)    # SHAP y casos individuales
X_ice = X_muestra.iloc[:200]                                       # ICE
X_fondo = X_train_completo.sample(100, random_state=SEMILLA)      # fondo de SHAP

# Umbral del 05: máximo F1 sobre predicciones de validación cruzada en entrenamiento
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
prob_cv = cross_val_predict(clone(modelo), X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
precision, recall, umbrales = precision_recall_curve(y_train, prob_cv)
f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
UMBRAL = umbrales[np.argmax(f1[:-1])]

prob_muestra = pd.Series(predecir_abandono(X_muestra), index=X_muestra.index, name='prob')
real_muestra = y.loc[X_muestra.index]
print(f'Umbral de decisión: {UMBRAL:.3f}  |  muestra de SHAP: {len(X_muestra)} clientes, '
      f'{real_muestra.sum()} abandonos ({real_muestra.mean() * 100:.1f}%)')

## 2. Gráficos ICE (Práctica 5, Actividad 1)

Una curva ICE muestra cómo cambia la probabilidad de abandono de **un** cliente si se varía una sola variable y se
dejan fijas las demás; hay una curva por cliente. Para usar `PartialDependenceDisplay` se envuelve la función de
predicción en un estimador mínimo de scikit-learn (código tomado de la ayuda *Gráfico ICE* del curso). La rejilla de
cada variable cubre del percentil 5 al 95; para `products_number` son sus valores posibles, de 1 a 4.

In [ ]:
class ClasificadorDesdeFuncion(ClassifierMixin, BaseEstimator):
    """Estimador mínimo que envuelve una función de predicción (ayuda "Gráfico ICE" del curso)."""
    def __init__(self, funcion=None):
        self.funcion = funcion
    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1])
        return self
    def predict_proba(self, X):
        p = self.funcion(X)
        return np.column_stack([1 - p, p])
    def predict(self, X):
        return (self.funcion(X) >= 0.5).astype(int)

clasificador = ClasificadorDesdeFuncion(predecir_abandono).fit(X_train_completo)

def rejilla(col, n=30):
    """Valores de la rejilla: percentiles 5 a 95 (o los valores posibles en products_number)."""
    if col == 'products_number':
        return np.array([1.0, 2.0, 3.0, 4.0])
    return np.linspace(X_ice[col].quantile(0.05), X_ice[col].quantile(0.95), n)

COLS_ICE = ['age', 'balance', 'products_number']
VALORES = {X_ice.columns.get_loc(c): rejilla(c) for c in COLS_ICE}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
PartialDependenceDisplay.from_estimator(clasificador, X_ice, COLS_ICE, kind='individual', custom_values=VALORES,
                                        ax=axes, ice_lines_kw={'alpha': 0.25, 'linewidth': 0.8})
fig.suptitle('Curvas ICE: probabilidad de abandono de 200 clientes de prueba', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_ice.png', bbox_inches='tight'); plt.show()

Para leer las curvas con cifras se calcula, en cada punto de la rejilla, el **promedio** de las 200 curvas (la
dependencia parcial) y la dispersión entre clientes.

In [ ]:
def resumen_ice(col, puntos):
    pd_res = partial_dependence(clasificador, X_ice, [col], kind='individual', custom_values={col: rejilla(col)})
    curvas = pd.DataFrame(pd_res['individual'][0], columns=pd_res['grid_values'][0])
    sel = [curvas.columns[np.argmin(np.abs(curvas.columns - p))] for p in puntos]
    return pd.DataFrame({'valor': sel, 'prob_media': curvas[sel].mean().to_numpy(),
                         'p10': curvas[sel].quantile(0.10).to_numpy(), 'p90': curvas[sel].quantile(0.90).to_numpy()})

for col, puntos in [('age', [25, 35, 45, 55, 65]), ('balance', [0, 50_000, 100_000, 150_000]),
                    ('products_number', [1, 2, 3, 4])]:
    print(f'--- {col} ---'); display(resumen_ice(col, puntos).round(3))

**Qué vemos:**

- **Edad: el efecto más fuerte, y no lineal.** La probabilidad media pasa del 10 % a los 25 años y el 13 % a los 35, al
  32 % a los 45 y al 48 % hacia los 55. El efecto cambia mucho entre clientes: a los 55 años, el 10 % de los clientes
  con menos riesgo queda por debajo del 11 %, y el 10 % con más riesgo por encima del 84 %. Las curvas no son
  paralelas, así que el efecto de la edad depende de las demás variables.
- **Saldo: casi plano.** La probabilidad media se mueve entre 17 % y 21 % en todo el rango; el modelo casi no lo usa.
- **Número de productos: forma de U.** Tener 2 productos se asocia al menor riesgo (10 %), 1 producto lo sube (27 %), y
  3 o 4 lo disparan (73 % y 86 %). En los datos hay pocos clientes con 3 o 4 productos (266 y 60 de 9.985, según el
  `03`), así que llevar a un cliente cualquiera a 4 productos crea combinaciones poco vistas por el modelo.

### 2.1 ICE centrado

Las curvas parten de alturas distintas porque cada cliente tiene un riesgo distinto. El ICE centrado le resta a cada
curva su primer valor, así que todas parten de 0 y muestran solo **cuánto cambia** la probabilidad. Si las curvas
centradas se abren en abanico, el efecto de la edad no es el mismo para todos.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
PartialDependenceDisplay.from_estimator(clasificador, X_ice, ['age'], kind='individual', centered=True,
                                        custom_values={X_ice.columns.get_loc('age'): rejilla('age')}, ax=ax,
                                        ice_lines_kw={'alpha': 0.25, 'linewidth': 0.8})
ax.set_title('ICE centrado de la edad (cambio de probabilidad desde la edad mínima de la rejilla)')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_ice_centrado_edad.png', bbox_inches='tight'); plt.show()

pd_c = partial_dependence(clasificador, X_ice, ['age'], kind='individual', custom_values={'age': rejilla('age')})
cambio = pd_c['individual'][0][:, -1] - pd_c['individual'][0][:, 0]
print(f'Cambio de probabilidad entre {rejilla("age")[0]:.0f} y {rejilla("age")[-1]:.0f} años: mediana {np.median(cambio):+.3f}, '
      f'percentil 10 {np.percentile(cambio, 10):+.3f}, percentil 90 {np.percentile(cambio, 90):+.3f}')

### 2.2 Curvas coloreadas por subgrupo: edad y miembro activo

Se repite el ICE de la edad coloreando las curvas según `active_member`, con la misma rejilla para los dos grupos
(código de la ayuda *Gráfico ICE*, sección "Colorear por subgrupo"). Si las formas difieren, las dos variables
interactúan.

In [ ]:
columna = 'age'
grupo = X_ice['active_member'] == 1
valores_edad = {X_ice.columns.get_loc(columna): rejilla(columna)}

fig, ax = plt.subplots(figsize=(7.5, 5))
display_ice = PartialDependenceDisplay.from_estimator(
    clasificador, X_ice[grupo], [columna], kind='individual', custom_values=valores_edad, ax=ax,
    ice_lines_kw={'color': 'tab:blue', 'alpha': 0.3, 'linewidth': 0.8})
PartialDependenceDisplay.from_estimator(
    clasificador, X_ice[~grupo], [columna], kind='individual', custom_values=valores_edad, ax=display_ice.axes_,
    ice_lines_kw={'color': 'tab:red', 'alpha': 0.3, 'linewidth': 0.8})
display_ice.axes_[0, 0].legend(handles=[Line2D([], [], color='tab:blue', label='miembro activo'),
                                        Line2D([], [], color='tab:red', label='miembro inactivo')])
display_ice.axes_[0, 0].set_title('ICE de la edad por actividad del cliente')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_ice_edad_actividad.png', bbox_inches='tight'); plt.show()

filas = []
for nombre, mascara in [('activo', grupo), ('inactivo', ~grupo)]:
    r = partial_dependence(clasificador, X_ice[mascara], ['age'], kind='average', custom_values={'age': rejilla('age')})
    curva = pd.Series(r['average'][0], index=r['grid_values'][0])
    filas.append({'grupo': nombre, 'clientes': int(mascara.sum()),
                  'prob_media_35': curva.iloc[np.argmin(np.abs(curva.index - 35))],
                  'prob_media_55': curva.iloc[np.argmin(np.abs(curva.index - 55))],
                  'max_prob_media': curva.max(), 'edad_del_max': curva.idxmax()})
display(pd.DataFrame(filas).set_index('grupo').round(3))

**Qué vemos:** sí hay **interacción entre edad y actividad**.

- **Activos:** la probabilidad media sube del 11 % a los 35 años al 30 % a los 55.
- **Inactivos:** sube del 15 % al 68 %, con un máximo del 74 % hacia los 57.

La edad pesa mucho más en los clientes inactivos. El ICE centrado muestra lo mismo: entre los 25 y los 58 años, la
mediana de los clientes gana 26 puntos de riesgo, pero el 10 % menos sensible gana solo 6 puntos y el 10 % más
sensible, 73. Esto explica parte de la ventaja del XGBoost sobre la regresión logística del `05`: un modelo lineal
asigna el mismo efecto de la edad a todos los clientes.

## 3. LIME (Práctica 5, Actividad 2)

LIME explica **una** predicción: genera miles de clientes parecidos al que se quiere explicar, los predice con el
modelo, pondera más a los más parecidos y ajusta un modelo lineal (el *sustituto*) que solo vale cerca de ese cliente.
Sus coeficientes son la explicación. Necesita la **función de predicción**, no el modelo, porque solo observa entradas
y salidas.

El explicador se crea con las variables originales de entrenamiento (sin faltantes) e indica cuáles son categóricas
(`products_number`, `credit_card`, `active_member`). Código de la ayuda *LIME* del curso, en modo clasificación.

In [ ]:
INDICES_CATEGORICAS = [VARIABLES_MODELO.index(c) for c in CATEGORICAS]

def predecir_dos_clases(X):
    p = predecir_abandono(X)
    return np.column_stack([1 - p, p])

explicador_lime = LimeTabularExplainer(X_train_completo.to_numpy(), feature_names=VARIABLES_MODELO,
                                       class_names=['se queda', 'abandona'], mode='classification',
                                       categorical_features=INDICES_CATEGORICAS, random_state=SEMILLA)

def explicar_lime(x, num_samples=5000, random_state=SEMILLA, num_features=6):
    explicador_lime.random_state = np.random.RandomState(random_state)
    return explicador_lime.explain_instance(x.to_numpy(), predecir_dos_clases, num_features=num_features,
                                            num_samples=num_samples)

### 3.1 Tres clientes: un acierto, un abandono no detectado y una falsa alarma

Con el umbral de decisión se clasifica a los 300 clientes de la muestra y se eligen tres casos:

- **Verdadero positivo (VP):** el cliente típico entre los que abandonan y el modelo detecta, es decir, el de
  probabilidad mediana dentro de ese grupo.
- **Falso negativo (FN):** el abandono que el modelo vio con **menos** riesgo; es el error más claro.
- **Falso positivo (FP):** el cliente que se queda y al que el modelo le dio **más** riesgo.

In [ ]:
pred_muestra = (prob_muestra >= UMBRAL).astype(int)
casos_vp = prob_muestra[(real_muestra == 1) & (pred_muestra == 1)]
casos_fn = prob_muestra[(real_muestra == 1) & (pred_muestra == 0)]
casos_fp = prob_muestra[(real_muestra == 0) & (pred_muestra == 1)]
CASOS = {
    'VP': (casos_vp - casos_vp.median()).abs().idxmin(),
    'FN': casos_fn.idxmin(),
    'FP': casos_fp.idxmax(),
}
print(f'En la muestra: VP={len(casos_vp)}, FN={len(casos_fn)}, FP={len(casos_fp)}')
tabla_casos = X_muestra.loc[list(CASOS.values())].copy()
tabla_casos.insert(0, 'caso', list(CASOS))
tabla_casos['prob_abandono'] = prob_muestra.loc[list(CASOS.values())].to_numpy()
tabla_casos['abandonó'] = real_muestra.loc[list(CASOS.values())].to_numpy()
display(tabla_casos.set_index('caso'))

In [ ]:
explicaciones_lime = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for ax, (caso, idx) in zip(axes, CASOS.items()):
    exp = explicar_lime(X_muestra.loc[idx])
    explicaciones_lime[caso] = exp
    pesos = exp.as_list(label=1)[::-1]
    ax.barh([p[0] for p in pesos], [p[1] for p in pesos], color=['#C44E52' if p[1] > 0 else '#4C72B0' for p in pesos])
    ax.axvline(0, color='k', lw=0.8)
    ax.set_title(f'{caso}: prob. del modelo {prob_muestra.loc[idx]:.2f} | sustituto {exp.local_pred[0]:.2f}')
    ax.set_xlabel('peso en la probabilidad de abandono')
fig.suptitle('LIME: las 6 condiciones más influyentes (rojo sube el riesgo, azul lo baja)', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RUTA_FIGS}/06_lime_casos.png', bbox_inches='tight'); plt.show()

for caso, exp in explicaciones_lime.items():
    print(f'--- {caso} ---')
    for condicion, peso in exp.as_list(label=1):
        print(f'  {condicion:35s} {peso:+.3f}')

**Qué vemos:**

- **VP** (50 años, inactivo, 1 producto, probabilidad 0,65). Lo empujan tener más de 44 años (+0,33), ser inactivo
  (+0,13) y tener un solo producto (+0,12). Es el perfil de riesgo que describen los gráficos ICE.
- **FN** (34 años, 2 productos, inactivo, probabilidad 0,04). Tener 2 productos (−0,17) y su edad (−0,13) pesan más
  que la inactividad (+0,11). Su perfil es el de los clientes que se quedan, y en estas 8 variables nada anticipaba
  su salida. Coincide con el `05`: los abandonos de clientes jóvenes con 2 productos son los que más se escapan.
- **FP** (51 años, inactivo, 1 producto, 10 años de antigüedad, probabilidad 0,68). Las mismas tres condiciones que en
  el VP y con pesos casi iguales; con estas variables es indistinguible de un cliente que abandona. Ese error no se
  corrige ajustando el modelo: requiere información que el conjunto no tiene, como transacciones, reclamos o uso de
  canales.

### 3.2 Fidelidad local

Si el sustituto lineal imita mal al modelo alrededor del cliente, la explicación no es confiable. Se compara la
predicción del modelo con la del sustituto (`local_pred`) y se revisa su R² local (`score`).

In [ ]:
fidelidad = pd.DataFrame([{'caso': caso, 'prob_modelo': prob_muestra.loc[CASOS[caso]],
                           'prob_sustituto': exp.local_pred[0], 'diferencia': exp.local_pred[0] - prob_muestra.loc[CASOS[caso]],
                           'R2_local': exp.score} for caso, exp in explicaciones_lime.items()]).set_index('caso')
display(fidelidad)

### 3.3 Estabilidad

Las perturbaciones son aleatorias, así que dos ejecuciones pueden dar explicaciones distintas. Se explica el caso VP
cinco veces con semillas distintas, primero con 1.000 perturbaciones y luego con 5.000, y se cuenta cuántas veces
aparece cada condición entre las tres más influyentes.

In [ ]:
filas = []
for num_samples in [1000, 5000]:
    for semilla in range(5):
        exp = explicar_lime(X_muestra.loc[CASOS['VP']], num_samples=num_samples, random_state=semilla)
        for rango, (condicion, peso) in enumerate(exp.as_list(label=1), start=1):
            filas.append({'num_samples': num_samples, 'semilla': semilla, 'rango': rango,
                          'condición': condicion, 'peso': peso})
estabilidad = pd.DataFrame(filas)
resumen_estab = (estabilidad.assign(en_top3=estabilidad['rango'] <= 3)
                 .groupby(['num_samples', 'condición'])
                 .agg(veces_en_top3=('en_top3', 'sum'), veces_en_top6=('rango', 'size'),
                      peso_medio=('peso', 'mean'), peso_desv=('peso', 'std'))
                 .sort_values(['num_samples', 'veces_en_top3', 'peso_medio'], ascending=[True, False, False]))
display(resumen_estab)

**Qué vemos:**

- **Fidelidad.** El sustituto se queda corto en el VP y el FP: predice 0,06 a 0,09 menos que el modelo, con un R² local
  de 0,55. En el FN imita peor (R² 0,29), así que esa explicación es la menos confiable. El sustituto es lineal y
  cerca de probabilidades muy bajas el modelo no lo es.
- **Estabilidad.** Las tres condiciones principales (edad > 44, inactivo, 1 producto) aparecen en el top 3 en las 10
  ejecuciones, con pesos que varían poco: desviación de hasta 0,012 con 1.000 perturbaciones y de hasta 0,005 con
  5.000. Las condiciones de peso pequeño cambian de orden e incluso de signo entre semillas; la antigüedad, por
  ejemplo, pasa de −0,012 a +0,001.

Conclusión: de una explicación de LIME solo conviene leer las condiciones principales, usando 5.000 perturbaciones.

### 3.4 Dos clientes con la misma probabilidad y perfiles distintos

¿El modelo llega a la misma probabilidad por las mismas razones? Se busca, entre los clientes de la muestra con riesgo
alto (probabilidad entre 0,4 y 0,7), el par con probabilidades casi iguales (diferencia menor a 0,02) y perfiles más
distintos (distancia entre sus variables estandarizadas).

In [ ]:
candidatos = prob_muestra[(prob_muestra > 0.4) & (prob_muestra < 0.7)].index
Z = (X_muestra.loc[candidatos] - X_train_completo.mean()) / X_train_completo.std()
mejor, mejor_dist = None, -1
for i, a in enumerate(candidatos):
    for b in candidatos[i + 1:]:
        if abs(prob_muestra[a] - prob_muestra[b]) < 0.02:
            dist = np.linalg.norm(Z.loc[a] - Z.loc[b])
            if dist > mejor_dist:
                mejor, mejor_dist = (a, b), dist
PAR = {'cliente A': mejor[0], 'cliente B': mejor[1]}
par = X_muestra.loc[list(PAR.values())].copy()
par.index = list(PAR)
par['prob_abandono'] = prob_muestra.loc[list(PAR.values())].to_numpy()
par['abandonó'] = real_muestra.loc[list(PAR.values())].to_numpy()
display(par)

for nombre, idx in PAR.items():
    exp = explicar_lime(X_muestra.loc[idx])
    print(f'--- {nombre} (prob. {prob_muestra[idx]:.3f}) ---')
    for condicion, peso in exp.as_list(label=1):
        print(f'  {condicion:35s} {peso:+.3f}')

**Qué vemos:** el cliente B es el VP de la sección 3.1; el A tiene 52 años, 2 productos, puntaje 452, 1 año de
antigüedad y es inactivo. Los dos tienen probabilidad 0,65, pero por caminos distintos:

- **Comparten** los aportes de la edad y la inactividad.
- **Se separan en los productos:** en B, tener un solo producto suma riesgo; en A, tener 2 productos le resta (−0,18).

LIME no muestra con claridad qué compensa esa resta en A. La sección 4 lo resuelve con SHAP, que reparte la
predicción exactamente. Además, A se quedó y B abandonó: la misma probabilidad no implica el mismo desenlace.